# 01. Integral norms: estimators and convergence

**Question:** given samples of a path, how should the integral norm be estimated?

---

## 0. Setting

### 0.1 Problem

Let $f:[0,T]\to\mathbb{R}^d$ be sampled at $t_0 < \cdots < t_N$: $N+1$ points, $N$ intervals. Estimate

$$\|f\|_{L^p} = \Big(\int_0^T |f|^p\Big)^{1/p} .$$

Write $g = |f|^p$ for the **integrand**, so the target and its estimator are

$$I(g) = \int_0^T g , \qquad Q_w(g) = \sum_{i=0}^N w_i\, g(t_i) .$$

The error bounds constrain $g$, not $f$. For odd $p$, $g$ has a kink at every zero of $f$ however smooth $f$ is.

The nodes are given, so the question is the choice of $w$. Two choices recur:

$$w^{\mathrm{trap}}_i = \tfrac12(\Delta t_{i-1} + \Delta t_i), \qquad w^{\mathrm{unif}}_i = \tfrac{T}{N+1},
\qquad \Delta t_i = t_{i+1}-t_i \ \ (0 \le i \le N-1) ,$$

with the convention $\Delta t_{-1} = \Delta t_N = 0$, so $w^{\mathrm{trap}}_0 = \tfrac12\Delta t_0$ and $w^{\mathrm{trap}}_N = \tfrac12\Delta t_{N-1}$. Each node carries the length of the interval for which it is nearest; the endpoints have interval on one side only. Both vectors sum to $T$.

Pointwise MSE is the uniform-weight rule:

$$\mathrm{MSE} = \frac{1}{N+1}\sum_i f(t_i)^2 = \tfrac1T\,Q^{\mathrm{unif}}(f^2) .$$

§2 and §3 ask when each converges to $I(g)$. In the loss application $f = \hat y - y$.

### 0.2 Relation to the baseline loss

On a **uniform** grid $w^{\mathrm{unif}}$ and $w^{\mathrm{trap}}$ differ only at the two endpoints (§1.2), so both converge and

$$\mathrm{MSE} \;\xrightarrow[N\to\infty]{}\; \frac1T\int_0^T f^2 \;=\; \frac1T\|f\|_{L^2}^2 .$$

The integral norm is the continuum limit of MSE, not a rival to it. §3 removes the uniformity.

### 0.3 Contents

All of it is standard. Verification is in `tests/`.

| section | | result |
|---|---|---|
| §1 | Choice of weights | trapezoid |
| §2 | Convergence rates | quadrature error is not a confounder |
| §3 | Non-uniform sampling | MSE converges to the wrong limit |
| §4 | Choice of $p$ | the discussion fixes $p=2$; the code does not |
| §5 | Summary | |

The ML literature on irregular sampling (Che et al., 2018; Rubanova et al., 2019; Shukla & Marlin, 2021; Kidger et al., 2020) modifies the model to ingest irregular observations and leaves the objective at pointwise MSE. §3 is the reason that matters.

> **Parallel with sFML.** There, a uniform average over a non-uniformly visited *state* space; here, over a non-uniformly sampled *time* axis.

## 1. Choice of weights

### 1.1 Quadrature rules

Each rule replaces $g$ on a subinterval by an interpolating polynomial and integrates that exactly.

| rule | interpolant | weights |
|---|---|---|
| left Riemann | degree 0, left endpoint | $w_i = \Delta t_i$, $w_N = 0$ |
| **trapezoid** | degree 1, both endpoints | $w_i = \tfrac12(\Delta t_{i-1} + \Delta t_i)$ |
| Simpson | degree 2, three points | $\tfrac{1}{3}\Delta t\,(1,4,2,\dots,4,1)$, uniform $\Delta t$ |
| Gauss-Legendre | chooses the nodes | weights at the roots of the Legendre polynomial $P_n$ |

Simpson's row carries a single spacing because it presupposes a uniform grid.

### 1.2 Degree of exactness

An interpolatory rule is exact for polynomials up to the degree of its interpolant (Davis & Rabinowitz, 1984, §2.1). Trapezoid: degree 1, by an argument local to each subinterval, hence valid on any grid.

Equal weights are exact on constants, and on lines only when the nodes are symmetric about the midpoint. `test_trapezoid_on_constants_and_lines`.

### 1.3 Error

For $g \in C^2$, with $h_{\max} = \max_i \Delta t_i$ (Davis & Rabinowitz, 1984),

$$\big|I(g) - Q_w(g)\big| \;\le\; \tfrac{T}{12}\,h_{\max}^2\,\|g''\|_\infty ,$$

by summing the local error $-\tfrac1{12}h_i^3 g''(\xi_i)$. Accuracy is limited by curvature and by the widest gap. §2 takes up $\|g''\|_\infty = \infty$.

### 1.4 Availability

| rule | exact to degree | requires |
|---|---|---|
| left Riemann | 0 | nothing |
| **trapezoid** | **1** | **nothing** |
| Simpson | 3 | uniform spacing, odd number of points |
| Gauss-Legendre | $2N-1$ | control of node positions |

Simpson's weights place the middle node at the midpoint; on unequal spacing the coefficients change per triple, so the standard formula becomes a different rule rather than an inaccurate one. Gauss-Legendre requires choosing where to sample.

With fixed nodes, **trapezoid is adopted**. Simpson and Gauss-Legendre may appear later as reference values on grids we control.

### 1.5 Cost: bias against variance

Weights chosen to remove a bias cost variance. Let $Y_i = g(t_i) + \varepsilon_i$ with $\operatorname{Var}(\varepsilon_i) = \sigma^2$ independent, and $\tilde w_i = w_i/\sum_j w_j$. Then

$$\operatorname{Var}\Big(\sum_i \tilde w_i Y_i\Big) = \sigma^2 \sum_i \tilde w_i^{\,2},
\qquad \sum_i \tilde w_i^{\,2} \ge \frac1{N+1} \quad\text{(Cauchy-Schwarz)},$$

with equality iff the weights are equal.

> Equal weights minimise variance. Trapezoid weights minimise bias. Nothing does both.

**Effective sample size.** An unweighted average of $\nu$ observations has variance $\sigma^2/\nu$. Setting $\sigma^2/\nu = \sigma^2\sum_i \tilde w_i^{\,2}$ gives Kish's

$$n_{\mathrm{eff}} = \frac{1}{\sum_i \tilde w_i^{\,2}} \le N+1 ,$$

the unweighted sample size of equal noise. It falls as weight concentrates: $N+1$ for equal weights, $1$ for all weight on one node. A clustered grid gives $n_{\mathrm{eff}} \ll N+1$, so the bias correction of §3 is paid for in variance.

Zhang & Wang (2016) show the preferable weighting reverses with sampling density; Godambe (1955) proved no best linear unbiased estimator exists even under simple random sampling.

Nothing in this notebook contains noise: §3 evaluates deterministic functions, so bias governs there.

Noise also biases. For $p=2$, $\mathbb{E}[Q_w(Y^2)] = \int_0^T f^2 + \sigma^2 T + O(h_{\max}^2)$, using $\sum_j w_j = T$. The term $\sigma^2 T$ is independent of $w$; removing it requires smoothing before integrating (Ramsay & Silverman, 2005, ch. 3 to 5).

### 1.6 Conclusion

$$\|f\|_{L^p}^p \;\approx\; \sum_{i=0}^N w^{\mathrm{trap}}_i\,|f(t_i)|^p ,$$

implemented as `integral_norm(t, x, p)` and `integral_distance(t, x, y, p)`.

Among rules assuming nothing about $f$ and no control of the nodes, trapezoid has the highest degree of exactness (§1.2) and is unbiased under any sampling design (§1.5). It is the default surviving the fewest assumptions, not an optimum: equal weights win on variance (§1.5), splines win given differentiability, smoothing wins given noise, and Godambe (1955) rules out an unconditional best.

§3 does not depend on the choice. Any consistent estimator of $\int f^2$ shows the same failure of equal weights.

Carried forward: adopt trapezoid; make no noise-robustness claim; report $n_{\mathrm{eff}}$ with every loss value.

## 2. Convergence rate (uniform grid)

§1 chose the weights. §2 fixes the sample budget, so §3 cannot be attributed to quadrature error.

The grid here is **uniform**, spacing $h = T/N$ over the $N$ intervals. Then $w^{\mathrm{unif}}$ and $w^{\mathrm{trap}}$ differ only at the endpoints, so the statements below cover both rules and the choice of weights is invisible. §3 removes the restriction and the two separate.

**Rates.** For $g \in C^2$, trapezoid is $O(h^2)$ and left Riemann $O(h)$ (Davis & Rabinowitz, 1984, §2.1 and §2.4). `test_convergence_rates_on_non_periodic_integrand` asserts both orders for the implementation.

**Euler-Maclaurin.** Truncating at $M$ terms,

$$\int_a^b g \;=\; h\Big[\tfrac12 g_0 + \cdots + \tfrac12 g_N\Big] \;-\; \sum_{k=1}^{M}\frac{B_{2k}}{(2k)!}\,h^{2k}\Big[g^{(2k-1)}(b) - g^{(2k-1)}(a)\Big] \;+\; R_M ,
\qquad |R_M| \;\le\; \frac{2\zeta(2M)}{(2\pi)^{2M}}\,h^{2M}\int_a^b \big|g^{(2M)}\big| ,$$

with $B_{2k}$ the Bernoulli numbers. Every retained term is a difference of odd derivatives at the endpoints, so for smooth periodic $g$ all vanish and only $R_M$ survives, for every $M$: convergence exceeds any power of $h$ (Trefethen & Weideman, 2014). A convergence test on a periodic integrand therefore measures nothing, which is why $e^t$ is used rather than $\sin^2$. `test_trapezoid_on_smooth_periodic_integrand`.

**Without $C^2$.** The bound is vacuous for nowhere-differentiable $g$, and an $\alpha$-Hölder bound gives only $O(N^{-\alpha})$, so $O(N^{-1/2})$ for Brownian $W$. But the per-interval errors are mean-zero Brownian bridge integrals of variance $\Theta(h^3)$, so $N$ independent terms sum to $\Theta(h^2)$ in variance and an error of order $N^{-1}$: one order worse than the smooth case, not two. Heuristic, not a proof.

**Budget.** A few hundred samples for smooth $g$, $10^3$ to $10^4$ for a rough one. Either way the quadrature error sits orders below the effect §3 measures, so it is controllable by sampling more densely. §3 turns to an error that is not.

## 3. Non-uniform sampling

§2 bounded $|Q_w(g) - I(g)|$ at finite $N$, which vanishes as the grid refines. This section concerns $\lim_m \mathrm{MSE} - \tfrac1T\int_0^T f^2$, which does not.

**Setup.** For $m = 1, 2, \dots$ let $\mathcal{T}_m = \{0 = t_0^{(m)} < \cdots < t_{N_m}^{(m)} = T\}$, with mesh $h_m = \max_i \Delta t_i^{(m)}$ and empirical measure $\mu_m = \frac{1}{N_m+1}\sum_i \delta_{t_i^{(m)}}$. The rules of §0.1 on this grid are

$$Q^{\mathrm{trap}}_m(g) = \sum_i w^{\mathrm{trap}}_i\, g\big(t_i^{(m)}\big),
\qquad
Q^{\mathrm{unif}}_m(g) = \frac{T}{N_m+1}\sum_i g\big(t_i^{(m)}\big) = T\!\int_0^T g\,\mathrm{d}\mu_m ,$$

with $\mathrm{MSE} = \tfrac1T Q^{\mathrm{unif}}_m(f^2)$. Both are instances of $Q_w$; only the weights differ.

§3.1 gives the two limits, §3.2 a density realising the hypothesis of the second, §3.3 a failure no weighting addresses, §3.4 the same result in Lebesgue form.

### 3.1 The two limits

**Proposition 1.** *Let $g \in C([0,T])$. If $h_m \to 0$ then $Q^{\mathrm{trap}}_m(g) \to \int_0^T g$.*

Consistency of the trapezoid rule for continuous integrands. The hypothesis constrains the mesh, not the placement of the nodes within it.

**Proposition 2.** *Let $g \in C([0,T])$ and $\mu_m \Rightarrow \mu$ weakly, $\mathrm{d}\mu = \rho\,\mathrm{d}t$ for a probability density $\rho$ on $[0,T]$. Then $Q^{\mathrm{unif}}_m(g) \to T\!\int_0^T g\rho$.*

Immediate from $Q^{\mathrm{unif}}_m(g) = T\!\int g\,\mathrm{d}\mu_m$. Both constructions of §3.2 satisfy the hypothesis.

**Corollary.** *Let $f \in C([0,T])$ and let both hypotheses hold. Then*

$$\underbrace{\tfrac1T\,Q^{\mathrm{trap}}_m(f^2)}_{\text{integral norm}} \;\longrightarrow\; \tfrac1T\!\int_0^T f^2 ,
\qquad
\underbrace{\tfrac1T\,Q^{\mathrm{unif}}_m(f^2)}_{\mathrm{MSE}} \;\longrightarrow\; \int_0^T f^2\rho ,$$

*the limits equal for every $f \in C([0,T])$ iff $\rho = 1/T$ a.e.*

Converse: if $\int_0^T f^2(\rho - \tfrac1T) = 0$ for every $f \in C([0,T])$, take $f = \sqrt g$ to get $\int_0^T g(\rho - \tfrac1T) = 0$ for every non-negative $g \in C([0,T])$, hence for every $g$ by linearity, hence $\rho = 1/T$ a.e.

> MSE integrates $f^2$ against the **sampling density**; the integral norm integrates it against **normalised Lebesgue measure**.

Accuracy is not what separates them. Left Riemann is first order and the crudest rule in §1.1, but its weights are the spacings, so Proposition 1 applies verbatim and it converges to $\tfrac1T\int f^2$ on a clustered grid too (`test_limits_under_non_uniform_sampling`). The divide is whether the weights see the grid.

The asymmetry between the hypotheses is the content. Proposition 1 requires only $h_m \to 0$, so $Q^{\mathrm{trap}}_m$ is consistent on every refining grid. Proposition 2 requires $\mu_m$ to converge, and the limit inherits whatever it converged to, so $Q^{\mathrm{unif}}_m$ is consistent only when $\mu_m \Rightarrow \mathrm{Unif}[0,T]$. Non-uniform nodes do not break quadrature: $\Delta t_i \approx 1/(N_m\rho(t_i))$ is exactly the correction $w^{\mathrm{trap}}$ applies. They break $w^{\mathrm{unif}}$, by a fixed amount rather than a vanishing one.

**A change of variables removes the bias.** Let $F$ be the CDF of $\rho$ and $u = F(t)$. Then $\int_0^T g\rho\,\mathrm{d}t = \int_0^1 (g\circ F^{-1})\,\mathrm{d}u$ and the nodes are uniform in $u$, so MSE is consistent in $u$-time. The bias is a coordinate artefact, removable when $\rho$ is known; reweighting is what remains when it is not.

### 3.2 A density realising the hypothesis

Proposition 2 needs $\mu_m \Rightarrow \rho\,\mathrm{d}t$. Independent draws from $\rho$ give one, by the strong law. So does the **quantile grid**: with $F$ the CDF of $\rho$,

$$t_i = F^{-1}(u_i), \qquad u_i = \tfrac{i}{N_m} ,$$

whose spacings satisfy $\Delta t_i = 1/\big(N_m\,\rho(t_i)\big) + O(N_m^{-2})$, realising $\rho$ to first order and without Monte Carlo noise. For a power-law $\rho$, $F^{-1}$ is a single power.

Take $\rho(t) = \tfrac1\alpha t^{1/\alpha - 1}$ on $[0,1]$, so $t_i = u_i^{\alpha}$, and $f(t) = ct$. Both limits are then closed-form and their ratio is

$$\frac{\int_0^1 f^2\rho}{\int_0^1 f^2} \;=\; \frac{3}{1 + 2\alpha} ,$$

independent of $c$ and unbounded as $\alpha \to 0$. `test_limits_under_non_uniform_sampling` checks both limits at $2^{20}$ points.

### 3.3 A failure independent of the weights

Proposition 1 requires $h_m \to 0$. A grid that places no node in an interval does not refine there, and the weights cannot compensate. The extreme case is exact.

**Proposition 3.** *If $\operatorname{supp} f \subseteq [a,b]$ and $\mathcal{T}_m \cap [a,b] = \emptyset$, then $Q^{\mathrm{trap}}_m(f^2) = Q^{\mathrm{unif}}_m(f^2) = 0$ while $\int_0^T f^2 > 0$.*

Both estimators evaluate $f$ only on $\mathcal{T}_m$, where it vanishes. No rule $\sum_i w_i f(t_i)^2$ with weights depending only on $\mathcal{T}_m$ separates such an $f$ from $0$.

So §3.1 and §3.3 are different failures. §3.1 misweights information that was collected, and a different $w$ removes it. Proposition 3 concerns information that was not collected, and no $w$ recovers it. In a table of loss values the two are indistinguishable, which is the argument for reporting the sampling density beside every loss.

Between Proposition 3 and full resolution lies partial sampling, covered by neither statement. Quantifying it needs the sweep in `docs/logbook/2026-08-09.md`.

> **Parallel with sFML.** A quantity differing between two models is invisible to the objective because the data does not separate them.

### 3.4 The same statement in Lebesgue form

Partitioning the range rather than the domain gives the layer cake identity: with $\mu_f(\lambda) = \big|\{t \in [0,T] : |f(t)| > \lambda\}\big|$,

$$\int_0^T |f|^p \;=\; \int_0^\infty p\,\lambda^{p-1}\,\mu_f(\lambda)\,\mathrm{d}\lambda ,$$

so $\|f\|_{L^p}$ depends on $f$ only through $\mu_f$, equivalently through the decreasing rearrangement $f^*$ (Lieb & Loss, 2001, §1.13 and ch. 3).

**It is the same estimator.** With $\hat\mu(\lambda) = \sum_i w_i \mathbb{1}[|f(t_i)| > \lambda]$,

$$\int_0^\infty p\lambda^{p-1}\hat\mu(\lambda)\,\mathrm{d}\lambda
\;=\; \sum_i w_i \int_0^{|f(t_i)|} p\lambda^{p-1}\,\mathrm{d}\lambda
\;=\; Q_w(|f|^p) ,$$

exactly. The weights survive: measuring how long $f$ spends above a level is itself an integral in $t$.

**It gives the shortest form of §3.1.** Both estimators are the same functional of a pushforward measure, differing in which measure is pushed forward:

$$\tfrac1T Q^{\mathrm{trap}}_m(f^2) \to \int |y|^2 \,\mathrm{d}\big(f_*\lambda_T\big)(y),
\qquad
\mathrm{MSE} \to \int |y|^2 \,\mathrm{d}\big(f_*\rho\big)(y) ,$$

with $\lambda_T$ normalised Lebesgue measure on $[0,T]$. The Corollary is then $f_*\lambda_T = f_*\rho$ for every $f$ iff $\rho = \lambda_T$.

**And a warning.** A norm seeing $f$ only through $\mu_f$ is invariant under every measure-preserving rearrangement of time: permute the path arbitrarily and $\|f\|_{L^p}$ is unchanged. Acceptable in a norm, disqualifying in a path-to-path loss, where order is the structure being learned. This is the reason the project does not stop at $L^p$, and what $p$-variation (notebook 02) is introduced to measure.

## 4. Choice of $p$

`integral_norm(t, x, p)` takes any $p \ge 1$, and $p = \infty$ for the sup norm. Nothing above fixes a value: §§1 to 3 hold for every $p$, with $g = |f|^p$ the integrand throughout.

On a probability space, and $[0,1]$ with Lebesgue measure is one, Lyapunov's inequality gives $\|f\|_{L^p} \le \|f\|_{L^q}$ for $p \le q$, with $\|f\|_{L^p} \to \sup_t |f(t)|$. Larger $p$ concentrates the loss on the worst-behaved part of the path. `test_lp_monotonicity_in_p`, `test_sup_norm`. The $T^{1/p}$ normalisation in `integral_distance` makes values comparable across $p$.

The discussion takes $p = 2$: it is the value comparable to MSE, and the only $p$ for which $L^p$ is a Hilbert space, needed once inner-product structure enters. $p = \infty$ is a useful diagnostic, and $p$ stays an argument rather than a constant.

## 5. Summary

$$\|f\|_{L^p}^p \;\approx\; Q^{\mathrm{trap}}_m\big(|f|^p\big) = \sum_{i=0}^N w^{\mathrm{trap}}_i\,|f(t_i)|^p , \qquad w^{\mathrm{trap}}_i = \tfrac12(\Delta t_{i-1} + \Delta t_i) ,$$

as `integral_norm(t, x, p)` and `integral_distance(t, x, y, p)`. Trapezoid because it has the highest degree of exactness available with the nodes given and no assumption on $f$ (§1). The error is $O(h^2)$ for $g \in C^2$ and about $N^{-1}$ without it, so it is controllable by sampling more densely (§2).

The one result not settled by a citation: for $f \in C([0,T])$, with $h_m$ and $\mu_m$ as in §3,

$$h_m \to 0 \;\Longrightarrow\; \tfrac1T Q^{\mathrm{trap}}_m(f^2) \to \tfrac1T\!\int_0^T f^2 ,
\qquad
\mu_m \Rightarrow \rho\,\mathrm{d}t \;\Longrightarrow\; \mathrm{MSE} \to \int_0^T f^2\rho ,$$

equal for every $f$ iff $\rho = 1/T$ a.e. The first hypothesis constrains the mesh, the second constrains where the nodes go. So MSE is **inconsistent** for $\tfrac1T\|f\|_{L^2}^2$ under non-uniform sampling and refining does not help. The bias is removable by the change of variables $u = F(t)$ when $\rho$ is known (§3.1); reweighting is what remains when it is not. §3.3 is a separate failure that no weighting addresses: if the grid misses where $f$ lives, every rule returns $0$.

**Provenance.** None of this is new. Quadrature on non-uniform grids is textbook (Davis & Rabinowitz, 1984; Trefethen & Weideman, 2014). The same weights are the *density compensation factors* of non-uniform FFT and MRI reconstruction, and $Q^{\mathrm{trap}}$ is the Horvitz-Thompson (1952) estimator of $\int f^2$ with MSE the unweighted one, so §3 restates the known bias of an unweighted estimator under unequal inclusion probabilities. §1.5 gives the reason not to overclaim it: by Zhang & Wang (2016) and Godambe (1955) the weights are a bias/variance tradeoff with no unconditional answer, and §3 is a regime where the bias dominates.

## Next steps

Everything above is standard and the verification lives in `tests/`. Three things follow.

1. **Train against it.** Port `integral_distance` to a differentiable torch loss in `src/pathloss/losses.py`, weights precomputed per batch, tested against the NumPy version. Then train a baseline on irregularly sampled data under MSE and under $Q^{\mathrm{trap}}$, and see whether §3's bias in the *evaluation* changes the *optimum*. It need not: an inconsistent objective can still have the right minimiser. That is the first genuinely open question here.
2. **$p$-variation** (notebook 02). Roughness rather than size, and not an integral. §3.4 is the reason it is needed: an $L^p$ norm cannot see the order of the path.
3. **Separation.** §3 says MSE converges to the wrong limit. A stronger objection is that no sampled loss separates paths at all, since two paths can differ between the nodes and agree on them, and refining fixes this pointwise but never uniformly. `docs/logbook/2026-08-10.md` and `2026-08-12.md` record the meeting proposal and a design: a self-consistency test for adequate sampling, and a similarity measure returning the region where two paths agree rather than a scalar.

Then: missingness in place of irregular sampling, and replacing §3.2's single density by a family indexed by how far $\rho$ is from uniform (`docs/logbook/2026-08-09.md`).

## References cited in this notebook

Keyed to `papers/references.bib`.

**Quadrature and numerical integration**
- **Davis & Rabinowitz (1984)**, *Methods of Numerical Integration*: error terms, Gauss-Legendre. §1, §2, §5.
- **Trefethen & Weideman (2014)**, *The exponentially convergent trapezoidal rule*, SIAM Review 56(3): the periodic degeneracy. §2, §5.

**Measure theory**
- **Lieb & Loss (2001)**, *Analysis*, 2nd ed.: layer cake representation §1.13, rearrangement ch. 3. §3.4.

**Weighting and estimation**
- **Horvitz & Thompson (1952)**, JASA 47(260): inverse-probability weighting. §5.
- **Godambe (1955)**, *JRSS B* 17(2): no best linear unbiased estimator exists. §1.5, §5.
- **Zhang & Wang (2016)**, *Ann. Statist.* 44(5): the efficiency of equal weighting reverses with sampling density. §1.5, §5.

**Functional data analysis**
- **Ferraty & Vieu (2006)**: discretised $L^2$ semi-metric, weights $w_j = t_j - t_{j-1}$. §1.
- **Ramsay & Silverman (2005)**, ch. 3 to 5: basis expansion, roughness penalties. §1.5. Route not yet taken.

**Irregular sampling in machine learning** (cited in §0.3 for what they leave alone)
- **Che et al. (2018)**, GRU-D, *Sci. Rep.* 8.
- **Rubanova, Chen & Duvenaud (2019)**, Latent ODEs, NeurIPS.
- **Shukla & Marlin (2021)**, mTAN, ICLR, and survey arXiv:2012.00168.
- **Kidger et al. (2020)**, Neural CDEs, NeurIPS.